# Model Validation Report
**Dataset**: UCI Adult (census income classification — predict whether annual income is >50K). A classic dataset for fairness analysis, with built-in `sex` / `race` fields.

**Validation framework**: the Federal Reserve's SR 11-7 (model risk management guidance).

**Four deliverables** (mapped to notebook sections):
1. Model performance → Section 3
2. Risk analysis → Sections 5 and 6
3. Governance recommendations → Section 7
4. Plain-language summary for non-technical managers → Section 8



## 0. Environment & data loading

In [3]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

COLS = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

def load_adult():
    try:  # prefer the local file in the same directory
        df = pd.read_csv('adult.data', header=None, names=COLS,
                        na_values=' ?', skipinitialspace=True)
        print('Loaded local adult.data')
    except FileNotFoundError:  # fallback: pull from OpenML
        from sklearn.datasets import fetch_openml
        df = fetch_openml('adult', version=2, as_frame=True).frame
        print('Loaded from OpenML')
    return df

df = load_adult()
print(df.shape)
df.head(3)


Loaded local adult.data
(32561, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


## 1. Data quality check
SR 11-7 pillar one: can the data behind the model be trusted?

In [4]:
print('Missing values (question marks):')
print((df == '?').sum()[(df == '?').sum() > 0])
print()
print('Duplicate rows:', df.duplicated().sum())
print()
print('Target distribution:')
print(df['income'].value_counts(normalize=True).round(3))
print()
print('Numeric columns describe:')
df.describe().T[['min','max','mean']].round(1)


Missing values (question marks):
workclass         1836
occupation        1843
native-country     583
dtype: int64

Duplicate rows: 24

Target distribution:
income
<=50K    0.759
>50K     0.241
Name: proportion, dtype: float64

Numeric columns describe:


,min,max,mean
age,17.0,90.0,38.6
fnlwgt,12285.0,1484705.0,189778.4
education-num,1.0,16.0,10.1
capital-gain,0.0,99999.0,1077.6
capital-loss,0.0,4356.0,87.3
hours-per-week,1.0,99.0,40.4


### ✍️ TODO-1: Data quality conclusions
- Which columns hold the missing values? How to handle them (drop / mode imputation / its own category)? Why?

  Missing values are concentrated in workclass (1,836), occupation (1,843), and native-country (583) — about 5.6% of rows, mostly overlapping. Kept as an explicit "unknown" category via one-hot encoding rather than dropped: the missingness itself may carry signal (e.g., never-worked), and 5.6% is too large to discard silently.
- Is the target imbalanced (>50K is ~24%)? What does that mean for validation?

  Target is imbalanced (24.1% >50K): accuracy is misleading — a dummy classifier scores 76%. Validation focuses on AUC, precision/recall, and F1 instead.
- Any suspicious outliers (e.g. hours-per-week = 99)?

  Outliers look plausible for self-reported census data (hours-per-week max 99, capital-gain max 99,999, age max 90). No impossible values; kept as-is and noted.


## 2. Modeling: baseline models
Train two models for comparison — a validation report must name its "model under review".

In [5]:
y = (df['income'] == '>50K').astype(int)
X = df.drop(columns=['income'])
num = X.select_dtypes(include='number').columns.tolist()
cat = X.select_dtypes(exclude='number').columns.tolist()
pre = ColumnTransformer([('num','passthrough',num),
                         ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
models = {'logreg': LogisticRegression(max_iter=1000),
          'rf': RandomForestClassifier(n_estimators=200, random_state=42)}
for name, m in models.items():
    m.fit(pre.fit_transform(X_train), y_train)
    print(name, 'trained')
print('Test set size:', len(y_test))


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg trained
rf trained
Test set size: 6513


## 3. Model performance
Deliverable 1: model performance.

In [6]:
Xt = pre.transform(X_test)
rows = []
for name, m in models.items():
    p = m.predict(Xt); s = m.predict_proba(Xt)[:,1]
    tn,fp,fn,tp = confusion_matrix(y_test,p).ravel()
    rows.append({'model':name,'accuracy':round(accuracy_score(y_test,p),3),
                 'AUC':round(roc_auc_score(y_test,s),3),
                 'precision':round(precision_score(y_test,p),3),
                 'recall':round(recall_score(y_test,p),3),
                 'F1':round(f1_score(y_test,p),3),
                 'FPR (false reject)':round(fp/(fp+tn),3),
                 'FNR (false accept)':round(fn/(fn+tp),3)})
perf = pd.DataFrame(rows)
perf


,model,accuracy,AUC,precision,recall,F1,FPR (false reject),FNR (false accept)
0,logreg,0.847,0.885,0.735,0.567,0.640,0.065,0.433
1,rf,0.857,0.908,0.734,0.636,0.682,0.073,0.364


### ✍️ TODO-2: Performance conclusions
- Which model is better, and on which metrics?

  Random forest wins overall: AUC 0.908 vs 0.885, recall 0.636 vs 0.567, F1 0.682 vs 0.640. Logistic regression has a marginally lower FPR (0.065 vs 0.073).

- If this were a credit-approval model, which costs more: FPR (rejecting good applicants) or FNR (approving bad ones)?

  As a credit model: FNR (approving a defaulter) costs the lender real money; FPR (rejecting a good applicant) costs opportunity and draws fair-lending scrutiny. RF's FNR of 0.364 vs 0.433 makes it the better fit where the lender's loss concentrates.
- Is the 0.5 threshold optimal? (Try 0.3 / 0.7 for comparison.)

  0.5 is not proven optimal — with a 24% positive rate, threshold tuning (e.g., 0.3/0.7) is the recommended next step to trade precision against recall. Note: the logreg baseline showed a convergence warning (unscaled features); it is reported as-is as an untuned baseline.


## 4. Stability / stress testing
Does the model hold up on data it hasn't seen?

In [7]:
rng = np.random.default_rng(42)
aucs = []
for _ in range(200):
    idx = rng.choice(len(y_test), len(y_test), replace=True)
    aucs.append(roc_auc_score(y_test.iloc[idx],
                models['rf'].predict_proba(Xt[idx])[:,1]))
print(f'AUC bootstrap 95% CI: [{np.percentile(aucs,2.5):.3f}, {np.percentile(aucs,97.5):.3f}]')
# Check performance drift across age bands
df_t = X_test.copy(); df_t['y']=y_test.values
df_t['pred']=models['rf'].predict(Xt); df_t['prob']=models['rf'].predict_proba(Xt)[:,1]
df_t['age_band']=pd.cut(df_t['age'],[0,30,45,60,100])
for band,g in df_t.groupby('age_band',observed=True):
    print(band,'n=%d'%len(g),'AUC=%.3f'%roc_auc_score(g['y'],g['prob']))


AUC bootstrap 95% CI: [0.900, 0.917]
(0, 30] n=2119 AUC=0.928
(30, 45] n=2463 AUC=0.881
(45, 60] n=1474 AUC=0.870
(60, 100] n=457 AUC=0.870


### ✍️ TODO-3: Stability conclusions
- Is the AUC confidence interval narrow? Is the model stable?

  AUC bootstrap 95% CI is [0.900, 0.917] — narrow (±0.009). The model is stable under resampling.
- Which age band deviates most, and what might explain it?

  Age-band drift: under-30 AUC is 0.928 vs 0.870–0.881 for older bands. Likely cause: incomes are more homogeneous early-career (mostly ≤50K), so the boundary is simpler there. Flagged for monitoring — performance is not uniform across age.


## 5. Fairness review ⭐ The highlight of this report
Do error rates differ systematically across groups?

**Important**: group differences ≠ proof of discrimination.
Explain possible causes and limitations — that is the difference between "validation" and "finger-pointing", and it scores points in a report.


In [8]:
def group_metrics(frame, by):
    out = []
    for gname, g in frame.groupby(by, observed=True):
        tn,fp,fn,tp = confusion_matrix(g['y'], g['pred']).ravel()
        out.append({by:gname,'n':len(g),
                    'selection_rate (>50K)':round((tp+fp)/len(g),3),
                    'TPR':round(tp/(tp+fn),3),
                    'FPR':round(fp/(fp+tn),3),
                    'AUC':round(roc_auc_score(g['y'],g['prob']),3)})
    return pd.DataFrame(out)

print('== By sex ==')
g_sex = group_metrics(df_t,'sex'); print(g_sex.to_string(index=False))
print()
print('== By race ==')
g_race = group_metrics(df_t,'race'); print(g_race.to_string(index=False))
# Disparate impact ratio: minority selection rate / majority selection rate
sr = g_sex.set_index('sex')['selection_rate (>50K)']
print('\nDisparate impact ratio (Female/Male):', round(sr['Female']/sr['Male'],3),
      '(<0.8 is typically a flag worth investigating)')
g_sex.to_csv('fairness_by_sex.csv',index=False)
g_race.to_csv('fairness_by_race.csv',index=False)
print('Saved fairness_by_sex.csv / fairness_by_race.csv')


== By sex ==
   sex    n  selection_rate (>50K)   TPR   FPR   AUC
Female 2158                  0.088 0.592 0.023 0.937
  Male 4355                  0.269 0.645 0.105 0.885

== By race ==
              race    n  selection_rate (>50K)   TPR   FPR   AUC
Amer-Indian-Eskimo   73                  0.082 0.444 0.031 0.892
Asian-Pac-Islander  200                  0.245 0.635 0.108 0.874
             Black  662                  0.107 0.591 0.028 0.935
             Other   45                  0.067 0.750 0.000 1.000
             White 5533                  0.222 0.640 0.080 0.905

Disparate impact ratio (Female/Male): 0.327 (<0.8 is typically a flag worth investigating)
Saved fairness_by_sex.csv / fairness_by_race.csv


### ✍️ TODO-4: Fairness conclusions
- Which group has the highest FPR, and by how much?

  Largest FPR gap is by sex: Male 0.105 vs Female 0.023 (4.5×). Selection rates: Male 0.269 vs Female 0.088 — disparate impact ratio 0.327, far below the 0.8 four-fifths guideline. This is a flag requiring investigation, not a verdict.By race: Black selection rate 0.107 vs White 0.222; "Other" AUC of 1.000 is a small-sample artifact (n=45), not a finding. Groups under n=100 cannot support firm conclusions.
- Possible causes (data bias? proxy features? historical structural factors?) — name at least 2

  Possible causes: (1) label bias — income reflects historical labor-market disparities, so the model learns a biased target; (2) proxy features — occupation, hours-per-week, and relationship status correlate with sex and carry historical patterns into the score; (3) representation — female >50K cases are rare in training, so the model is conservative on that group.
- Limitations: what can this dataset NOT prove? (correlation ≠ causation; the labels themselves may be biased)

  Limitations: correlation ≠ causation; 1994 census data is stale for today's decisions; "accurate" predictions can still be unfair if the labels embed past discrimination.
- Mitigation: per-group threshold tuning / debiased feature engineering / human review of high-risk decisions

  Mitigation: per-group threshold tuning, proxy-feature review with re-testing, human review of borderline and high-stakes decisions, and ongoing monitoring of selection-rate ratios.


## 6. Risk rating (SR 11-7 structure)
Rate the model on SR 11-7's three dimensions:

| Dimension | Rating (Low/Med/High) | One-line rationale |
|---|---|---|
| Conceptual soundness | (you fill) |  |
| Data quality | (you fill) |  |
| Ongoing monitoring | (you fill) |  |

### ✍️ TODO-5: Overall risk rating + one-sentence verdict
- Conceptual soundness: **Medium** — standard classifiers for a binary task, but the target bakes historical labor-market bias into the objective; the model is a correlational scorer, not a causal tool.
- Data quality: **Medium** — 5.6% missingness handled as its own category, 24 duplicates negligible, but 1994 census data is dated for current deployment.
- Ongoing monitoring need: **High** — material fairness flags require continuous group-metric tracking.
- **Overall: Medium-High. Deploy only with conditions** — fairness monitoring, per-group thresholds, and human review of high-stakes decisions.


## 7. Governance recommendations
### ✍️ TODO-6: Monitoring & governance
- Which metrics to monitor post-deployment, and how often?
- What triggers a re-validation?
- Who owns model decisions (human-review mechanism)?

1. **Monitor monthly**: overall AUC plus per-group selection rates, TPR, and FPR by sex and race. Alert if the disparate-impact ratio falls below 0.8 or any group's FPR shifts more than 3 percentage points.
2. **Re-validate when**: the input data distribution shifts (new applicant mix, new data source), the model is retrained, the decision threshold changes — or any fairness guardrail is breached two months in a row.
3. **Human in the loop**: borderline scores (predicted probability 0.40–0.60) and every high-stakes decision require human sign-off. The model recommends; a person decides.
4. **Clear ownership**: one named model owner in the business unit, plus an independent validator. Full re-validation at least annually, per SR 11-7.
5. **Audit trail**: version every change to data, model, and thresholds; log all human overrides with reasons.


## 8. Summary for non-technical managers 📋
### ✍️ TODO-7: Explain it in 5 sentences

- What does this model do?

  This model predicts whether a person's annual income is above $50K — a standard income-classification scoring task.
- How does it perform (one sentence)?

  It is right 86% of the time (AUC 0.91) and its results hold up under repeated testing.
- What is the biggest risk?

  The biggest risk is uneven treatment across groups: women are selected at less than one-tenth the rate of men (disparate-impact ratio 0.33) — in a real lending deployment, that is a fair-lending problem.
- Who might it treat unfairly, and what will we do about it?

  Women and Black applicants face the largest gaps; we will tune decision thresholds per group, review proxy features, and require human review of high-stakes decisions.
- Recommendation: deploy / deploy with conditions / do not deploy?

  **Recommendation: do not deploy as-is — deploy only with conditions** (fairness monitoring, per-group thresholds, human oversight).

---

